<!-- notebook-header -->
<a id="top"></a>

# 10. Matrices, Math & Bits

*LeetCode Playbook · notebook 10 of 12* · Index arithmetic on grids, bit arrays and bit tricks, and the math recipes interviews use.

**In this notebook:** [Matrices](#s21) · [Math, Bits & Geometry](#s22)

← [Graphs](09_Graphs.ipynb) · [All notebooks](01_Start_Here.ipynb#notebooks) · [A-Z problem finder](12_Interview_Day.ipynb#finder) · [Design Problems](11_Design.ipynb) →

Run the setup cell below first; after that every section runs from its first cell.

In [ ]:
# Setup: everything this notebook imports. Run this cell first.
from collections import Counter, defaultdict, deque, OrderedDict
from functools import cmp_to_key, lru_cache
from itertools import accumulate, combinations, permutations, product
import bisect, heapq, math, random, string

<a id="s21"></a>

## Matrices

> A grid is a list of rows, and every matrix trick is index arithmetic on `(r, c)`: the row first, counting down; the column second, counting right.

**Reach for it when** the input is a grid, board or image and the problem says *in place* or *O(1) extra space*, *rotate*, *spiral* or *clockwise*, *diagonal*, *set its entire row and column*, *every row and every column is sorted*, or *all cells update at the same time*. (Searching a grid for regions or shortest paths is BFS/DFS: see [Graphs I](09_Graphs.ipynb#s17). This section is about moving indices around.)

**In this repo:** `math_geometry/` (Rotate Image, Spiral Matrix, Set Matrix Zeroes) · basics: `practice/simple/basics/matrices/01_rotate_image.py`, `02_spiral_matrix.py`, `03_set_matrix_zeroes.py`, `04_search_2d_matrix_staircase.py`, `05_game_of_life_in_place.py`, `06_matrix_traversal_patterns.py` · grid walks with bounds checks in the bank: `practice/simple/39_word_search.py`, `practice/simple/41_number_of_islands.py`, `practice/simple/44_rotting_oranges.py` · diagonals as `r - c` and `r + c`: `practice/simple/40_n_queens.py`

### The picture

```text
grid = [[1,  2,  3,  4],          R = len(grid)    = 3 rows
        [5,  6,  7,  8],          C = len(grid[0]) = 4 columns
        [9, 10, 11, 12]]

          c=0  c=1  c=2  c=3
   r=0  [  1    2    3    4 ]     grid[1][2] == 7: the row first (down), then the column (right)
   r=1  [  5    6   (7)   8 ]     its 4 neighbours: up (0, 2), down (2, 2), left (1, 1), right (1, 3)
   r=2  [  9   10   11   12 ]     a step is (r + dr, c + dc); it is inside iff 0 <= r < R and 0 <= c < C

   "\" diagonals: r - c is constant          "/" anti-diagonals: r + c is constant
          0  -1  -2  -3                             0   1   2   3
          1   0  -1  -2                             1   2   3   4
          2   1   0  -1                             2   3   4   5
```

The main walk of this section, the spiral, peels the grid like an onion: the outer ring first, then the ring inside it.

```text
            left                  right
             v                      v
   top  ->   1 -->  2 -->  3 -->    4       lap 1: top row ->, right column down, bottom row <-,
                                    |              left column up. Each wall steps inward right
             5 -->  6 -->  7        8              after its side is walked, so the next side starts
             ^                      |              one cell later and no corner is read twice.
 bottom ->   9 <-- 10 <-- 11 <--   12       lap 2: only row 1, columns 1..2 is left: 6, 7
```

**Why it is cheap:** reading every cell is already O(R·C), so most matrix problems are not about time. Their brute forces waste memory, and each optimal version trades that memory for one observation:

| Problem | What the brute force wastes | The observation that removes it |
|---|---|---|
| Rotate Image | a second matrix | a quarter-turn is two mirror flips, and a flip is only swaps |
| Spiral Matrix | a `visited` grid | the unvisited cells always form a rectangle, so four numbers describe them |
| Set Matrix Zeroes | a copy, or two sets | a row that holds a zero is doomed anyway, so its first cell can carry the flag |
| Game of Life | a copy of the board | a 0/1 cell has spare bits to hold tomorrow's value |
| Search a sorted matrix | time: an O(R·C) scan | from a corner, one comparison drops a whole row or column: O(R + C) |

### From idea to code

Every tool in this section is index arithmetic, so first the vocabulary: directions, the bounds check, and making a grid without the classic aliasing bug.

In [ ]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]             # up, down, left, right


def neighbors(grid, r, c):
    R, C = len(grid), len(grid[0])
    out = []
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < R and 0 <= nc < C:                # check BEFORE reading grid[nr][nc]
            out.append((nr, nc))
    return out


grid = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]
print(neighbors(grid, 0, 0), neighbors(grid, 1, 2))   # [(1, 0), (0, 1)] [(0, 2), (2, 2), (1, 1), (1, 3)]

shared = [[0] * 3] * 2                 # WRONG: a list holding the SAME row twice
fresh = [[0] * 3 for _ in range(2)]    # right: a new row each time round the loop
shared[0][0] = fresh[0][0] = 7
print(shared, fresh)                   # [[7, 0, 0], [7, 0, 0]] [[7, 0, 0], [0, 0, 0]]
clone = [row[:] for row in grid]       # copy a grid row by row
clone[0][0] = 99
print(grid[0][0])                      # 1: the original is untouched

**Try it**
- Delete the bounds check in `neighbors` and print `[grid[nr][nc] for nr, nc in neighbors(grid, 0, 0)]`: no error, you get `[9, 5, 4, 2]`. Python reads `grid[-1][0]` as the *last* row, so a missing check silently wraps around. Only the far side crashes, and only when you read the cells: `[grid[nr][nc] for nr, nc in neighbors(grid, 2, 3)]` raises `IndexError` at `(3, 3)`.
- Replace the copy with `clone = grid[:]` and rerun: `grid[0][0]` prints 99. `grid[:]` copies the outer list only; both lists still point at the same rows.
- Make an 8-direction list `DIRS8 = DIRS + [(-1, -1), (-1, 1), (1, -1), (1, 1)]` and loop over it instead: the corner `(0, 0)` has 3 neighbours, the inner cell `(1, 1)` has 8.

Now the walk with the most index decisions in it: the spiral.

**The idea in one sentence:** *the cells you have not visited yet always form a rectangle, so walk its four sides in order and move each wall inward right after you walk it.*

| Decision | Spiral (four walls) answer |
|---|---|
| **State**: what must I remember? | four walls `top, bottom, left, right`, and the output list |
| **Definition**: what exactly does each variable mean? | the unvisited cells are exactly rows `top..bottom` × columns `left..right`, **all four inclusive** |
| **Invariant**: what is true at the end of every step? | the unvisited cells form a rectangle, possibly empty: it is empty exactly when `top > bottom` or `left > right`. Every cell outside it has been output once, in spiral order |
| **Step**: how does one side change the state? | walk one side, then move that wall in by one. A side is a row or a column, and its line must still exist: a row needs `top <= bottom`, a column needs `left <= right`. The `while` test guarantees both for the first two sides. But `top += 1` can use up the last row and `right -= 1` the last column, so re-check `top <= bottom` before the bottom row and `left <= right` before the left column |
| **Record**: when is the answer updated? | append each cell as you walk past it, *before* that side's wall moves |
| **Init**: starting values | `top, bottom, left, right = 0, R - 1, 0, C - 1`: the whole grid is unvisited |
| **Return**: what comes back? | the output once the walls cross; `[]` for an empty grid |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "how many rows, how many columns" | `R, C = len(grid), len(grid[0])` (after checking the grid is not empty) |
| "the cell in row r, column c" | `grid[r][c]`: row first, always |
| "one step in every direction, if it stays inside" | `for dr, dc in DIRS:` then `if 0 <= nr < R and 0 <= nc < C:` |
| "walk the top wall, left to right" | `for c in range(left, right + 1): out.append(grid[top][c])` |
| "that wall is used up" | `top += 1` |
| "walk the right column, top to bottom" | `for r in range(top, bottom + 1): out.append(grid[r][right])`, after `top += 1` |
| "walk the bottom wall, right to left" | `for c in range(right, left - 1, -1):` (the stop `left - 1` is excluded) |
| "walk the left column, bottom to top" | `for r in range(bottom, top - 1, -1):` |
| "is there still a row between the walls?" | `if top <= bottom:` |
| "is there still a column between the walls?" | `if left <= right:` |
| "the walls have crossed" | `while top <= bottom and left <= right:` is false |

On every side the order is RECORD, then STEP: read the side while its wall still marks it, then move the wall. Moving it first skips that row; moving it one side too late reads the corner twice.

In [ ]:
def spiral_order(grid):
    if not grid or not grid[0]:
        return []
    out = []                                         # STATE: the cells so far, in spiral order
    top, bottom = 0, len(grid) - 1                   # STATE + INIT: unvisited = rows top..bottom,
    left, right = 0, len(grid[0]) - 1                #   columns left..right (all inclusive)
    while top <= bottom and left <= right:           # the rectangle is not empty
        for c in range(left, right + 1):             # top row, left -> right
            out.append(grid[top][c])                 # RECORD as we walk past
        top += 1                                     # STEP: the top wall moves in
        for r in range(top, bottom + 1):             # right column, going down
            out.append(grid[r][right])               # RECORD
        right -= 1                                   # STEP: the right wall moves in
        if top <= bottom:                            # a row is still left
            for c in range(right, left - 1, -1):     # bottom row, right -> left
                out.append(grid[bottom][c])          # RECORD
            bottom -= 1                              # STEP: the bottom wall moves in
        if left <= right:                            # a column is still left
            for r in range(bottom, top - 1, -1):     # left column, going up
                out.append(grid[r][left])            # RECORD
            left += 1                                # STEP: the left wall moves in
    return out                                       # RETURN: the walls have crossed


print(spiral_order(grid))                            # [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]
print(spiral_order([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))   # [1, 2, 3, 6, 9, 8, 7, 4, 5]

**Try it**
- Delete the line `if top <= bottom:` (keep the loop under it, dedented) and run `spiral_order([[1, 2, 3]])`: `[1, 2, 3, 2, 1]`. With one row left, the "bottom row" is the top row you already walked, walked back.
- Do the same with `if left <= right:` and run `spiral_order([[1], [2], [3]])`: `[1, 2, 3, 2]`. The single column is walked down and then partly back up.
- Change `range(right, left - 1, -1)` to `range(right, left, -1)`: on `grid` the 9 goes missing. A `range` never includes its stop, so walking *down to* `left` needs `left - 1`.
- Print `(top, bottom, left, right)` at the top of the `while` for a 4 x 4 grid: `(0, 3, 0, 3)` then `(1, 2, 1, 2)`, one lap per ring.

### Watch it work

Each block is one lap; each line is one side of the current rectangle. Watch the guards skip the sides that no longer exist.

In [ ]:
def trace_spiral(grid):
    top, bottom, left, right = 0, len(grid) - 1, 0, len(grid[0]) - 1
    while top <= bottom and left <= right:
        print(f"walls: rows {top}..{bottom}, columns {left}..{right}")
        print("   top row    ->", [grid[top][c] for c in range(left, right + 1)])
        top += 1
        print("   right col  v ", [grid[r][right] for r in range(top, bottom + 1)])
        right -= 1
        if top <= bottom:
            print("   bottom row <-", [grid[bottom][c] for c in range(right, left - 1, -1)])
            bottom -= 1
        else:
            print("   bottom row    skipped: no row left")
        if left <= right:
            print("   left col   ^ ", [grid[r][left] for r in range(bottom, top - 1, -1)])
            left += 1
        else:
            print("   left col      skipped: no column left")


trace_spiral(grid)

**Try it**
- Predict, then run `trace_spiral([[1, 2], [3, 4], [5, 6], [7, 8]])`: a single lap, `1 2 | 4 6 8 | 7 | 5 3`. After it `left = 1 > right = 0`, so the walls have crossed.
- Run `trace_spiral([[5]])`: the top row takes the only cell, and both guards skip their side.
- Move `top += 1` below the right-column `print` (the wall moves one side too late): the right column becomes `[4, 8, 12]`, so the corner 4 is read twice. The same edit in `spiral_order` gives `[1, 2, 3, 4, 4, 8, 12, 11, 10, 9, 5, 6, 7, 7]`.
- Run it on a 5 x 5 grid, `[[5 * r + c for c in range(5)] for r in range(5)]`: three laps, and the last one is just the centre cell 12.

### The rest of the toolbox

**Derive any index map in 30 seconds.** Say where `(0, 0)` and `(0, 1)` must land, then read the formula off. Clockwise: the top-left corner goes to the top-right corner of the new grid, `(0, R-1)`, and one step right becomes one step down, so `(r, c) -> (c, R-1-r)`. Always check a map on a non-square grid, where a wrong `R` or `C` shows up at once.

| Move | `(r, c)` goes to | New shape | A new grid in one line |
|---|---|---|---|
| transpose | `(c, r)` | C x R | `[list(t) for t in zip(*g)]` |
| rotate clockwise | `(c, R-1-r)` | C x R | `[list(t) for t in zip(*g[::-1])]` |
| rotate counter-clockwise | `(C-1-c, r)` | C x R | `[list(t) for t in zip(*g)][::-1]` |
| rotate 180° | `(R-1-r, C-1-c)` | R x C | `[row[::-1] for row in g[::-1]]` |
| mirror left-right | `(r, C-1-c)` | R x C | `[row[::-1] for row in g]` |

| Name for a cell | Key | All cells with one key |
|---|---|---|
| flat index | `i = r * C + c` | back: `r, c = divmod(i, C)` |
| `\` diagonal | `k = r - c` | `for r in range(max(0, k), min(R, C + k)):` with `c = r - k` |
| `/` anti-diagonal | `d = r + c` | `for r in range(max(0, d - C + 1), min(R, d + 1)):` with `c = d - r` (keeps `0 <= c < C`) |
| ring (layer) | `min(r, c, R-1-r, C-1-c)` | its distance to the nearest edge |
| 3 x 3 box (sudoku) | `(r // 3) * 3 + c // 3` | boxes numbered 0..8, row by row |

#### Rotate an n x n matrix in place

**Idea:** a clockwise quarter-turn sends `(r, c)` to `(c, n-1-r)`. That move is two mirror flips in a row, and a flip is only swaps: mirror across the main diagonal (the transpose, `(r, c) -> (c, r)`), then mirror left-right (reverse each row, `(c, r) -> (c, n-1-r)`).

```text
  1 2 3                      1 4 7                        7 4 1
  4 5 6   -- transpose -->   2 5 8   -- reverse rows -->  8 5 2
  7 8 9    swap across "\"   3 6 9     flip left-right    9 6 3
```

The other classic way turns four cells at once, ring by ring. Call them `a, b, d, e` (the name `c` is taken: it is the column). Each one is the previous one with the same map applied: `b` is where `a` goes, `d` is where `b` goes, `e` is where `d` goes, and `e` goes back to `a`:

```text
   n = 4, ring 0, c = 1:      . a . .        a = (0, 1) -> b = (1, 3)
                              . . . b        b = (1, 3) -> d = (3, 2)
                              e . . .        d = (3, 2) -> e = (2, 0)
                              . . d .        e = (2, 0) -> a
```

In the code, every cell on the left receives the value of the cell that turns into it, so the right side is the left side shifted one place, last first: `a, b, d, e` receive `e, a, b, d`.

In [ ]:
def rotate_clockwise(m):                       # n x n, in place
    n = len(m)
    for r in range(n):
        for c in range(r + 1, n):              # above the diagonal only: each pair swapped once
            m[r][c], m[c][r] = m[c][r], m[r][c]
    for row in m:
        row.reverse()                          # then mirror left <-> right


def rotate_by_rings(m):                        # the same turn, four cells per move
    n = len(m)
    for r in range(n // 2):                    # ring r (the centre of an odd n stays put)
        for c in range(r, n - 1 - r):          # the ring's top edge, minus its last cell
            m[r][c], m[c][n - 1 - r], m[n - 1 - r][n - 1 - c], m[n - 1 - c][r] = \
                m[n - 1 - c][r], m[r][c], m[c][n - 1 - r], m[n - 1 - r][n - 1 - c]


a = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
b = [row[:] for row in a]
rotate_clockwise(a)
rotate_by_rings(b)
print(a)         # [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
print(a == b)    # True

**Try it**
- Change `range(r + 1, n)` to `range(n)` in the transpose: you get `[[3, 2, 1], [6, 5, 4], [9, 8, 7]]`, only the mirror image. Every pair was swapped twice, which undid the transpose.
- Replace `for row in m: row.reverse()` with `m.reverse()` (reverse the *order* of the rows): `[[3, 6, 9], [2, 5, 8], [1, 4, 7]]`, the counter-clockwise turn.
- Call `rotate_clockwise` four times on the same matrix and compare it with a saved copy: four quarter-turns change nothing.
- In `rotate_by_rings`, print `(r, c)` for each move on a 4 x 4: `(0, 0) (0, 1) (0, 2) (1, 1)`. Four moves of four cells cover all 16.

#### Set Matrix Zeroes with O(1) extra space

**Idea:** the answer needs only R + C facts ("row r has a zero", "column c has a zero"), and the matrix's own first column and first row can hold them. We write `m[r][0] = 0` only when row r contains a zero, and then that whole row ends up zero anyway, so nothing is lost (the same goes for `m[0][c]` and column c). The markers do overwrite two facts, whether row 0 and column 0 had a zero *themselves*: save those in two booleans first, and fix row 0 and column 0 last.

```text
   row flags live in column 0, column flags live in row 0

     1  1  1  1                  1  0  1  1      m[0][1] = 0: "zero column 1"
     1  0  1  1   -- mark -->    0  0  1  1      m[1][0] = 0: "zero row 1"
     1  1  1  1                  1  1  1  1
```

In [ ]:
def set_zeroes(m):
    R, C = len(m), len(m[0])
    row0 = any(x == 0 for x in m[0])                  # STATE: the two facts the markers overwrite
    col0 = any(m[r][0] == 0 for r in range(R))
    for r in range(1, R):                             # mark: row flags in column 0,
        for c in range(1, C):                         #       column flags in row 0
            if m[r][c] == 0:
                m[r][0] = m[0][c] = 0                 # STEP: flag row r and column c
    for r in range(1, R):                             # apply the flags to the inner cells
        for c in range(1, C):
            if m[r][0] == 0 or m[0][c] == 0:
                m[r][c] = 0                           # RECORD: apply the flags
    if row0:                                          # the marker row and column go LAST
        for c in range(C):
            m[0][c] = 0                               # RECORD: row 0
    if col0:
        for r in range(R):
            m[r][0] = 0                               # RECORD: column 0


z = [[0, 1, 2, 0], [3, 4, 5, 2], [1, 3, 1, 5]]
set_zeroes(z)
print(z)    # [[0, 0, 0, 0], [0, 4, 5, 0], [0, 3, 1, 0]]

**Try it**
- Print `m` right after the marking loop for `[[1, 1, 1], [1, 0, 1], [1, 1, 1]]`: `[[1, 0, 1], [0, 0, 1], [1, 1, 1]]`. The flags sit in row 0 and column 0, exactly as in the picture.
- Start the two apply loops at 0 (`range(R)` and `range(C)`) and rerun the example: every cell becomes 0. Row 0 is applied first, and its new zeros then look like flags for every column.
- Try the shortcut that drops the booleans: mark from every cell (`range(R)` and `range(C)` in the marking loops), then let `if m[0][0] == 0:` zero both row 0 and column 0. `[[1, 0], [1, 1]]` gives `[[0, 0], [0, 0]]` instead of `[[0, 0], [1, 0]]`: one cell cannot say "row 0" and "column 0" separately.
- Delete `col0` and its block, then run `[[1, 1], [0, 1]]`: you get `[[1, 1], [0, 0]]` instead of `[[0, 1], [0, 0]]`. The original 0 still works as row 1's flag, but nothing ever zeroes column 0.

#### Search a matrix sorted by rows and by columns: the staircase

**Idea:** stand on the top-right corner. It is the largest value in its row and the smallest in its column, so one comparison throws away a whole column (too big: everything below it is even bigger) or a whole row (too small: everything to its left is even smaller). After each step the cells still in play are rows `r..` and columns `..c`, and `(r, c)` is again their top-right corner, so the same argument repeats.

```text
  target = 5                    15 > 5: the column below is all bigger   -> step left
   1   4   7  11  15            11 > 5, then 7 > 5: same                  -> left, left
   2   5   8  12  19             4 < 5: the row to its left is smaller   -> step down
   3   6   9  16  22             5 == 5: found after 5 looks (never more than R + C - 1)
  10  13  14  17  24
  18  21  23  26  30
```

In [ ]:
def search_sorted_matrix(m, target):
    if not m or not m[0]:
        return False
    r, c = 0, len(m[0]) - 1                    # top-right corner
    while r < len(m) and c >= 0:               # still inside
        if m[r][c] == target:
            return True
        if m[r][c] > target:
            c -= 1                             # everything below is bigger: drop the column
        else:
            r += 1                             # everything to the left is smaller: drop the row
    return False


sm = [[1, 4, 7, 11, 15], [2, 5, 8, 12, 19], [3, 6, 9, 16, 22], [10, 13, 14, 17, 24], [18, 21, 23, 26, 30]]
print(search_sorted_matrix(sm, 5), search_sorted_matrix(sm, 20))   # True False

**Try it**
- Count the looks for target 20, which is absent: 9 = R + C − 1. The walk goes 15, 19, 22, 16, 17, 26, 23, 21, 18 and steps off the bottom.
- Start at the bottom-left corner instead (`r, c = len(m) - 1, 0`, move up when too big and right when too small): same answers. That corner is also the biggest of one line and the smallest of the other. The top-left is not: both of its moves go to bigger values.
- Search for 0 and for 31: the walk leaves the grid on the left side and on the bottom side, and both return `False`.

#### Game of Life in place: two states in one cell

**Idea:** every cell must be computed from the *old* board, so writing a new value straight away corrupts neighbours that are read later. A cell needs one bit but an int has plenty: keep today in bit 0, write tomorrow into bit 1, and let neighbours read `cell & 1`. A last pass shifts tomorrow into place. (For values wider than one bit, pack two numbers in base K: `cell = old + K * new`, read the old value with `% K`, and finish with `// K`.)

```text
  value   bit 1 (tomorrow)   bit 0 (today)
    0           0                 0          dead, stays dead
    1           0                 1          alive, dies
    2           1                 0          dead, is born
    3           1                 1          alive, survives
```

In [ ]:
def game_of_life(board):
    R, C = len(board), len(board[0])
    for r in range(R):
        for c in range(C):
            live = 0
            for dr in (-1, 0, 1):
                for dc in (-1, 0, 1):
                    nr, nc = r + dr, c + dc
                    if (dr or dc) and 0 <= nr < R and 0 <= nc < C:   # 8 neighbours, inside
                        live += board[nr][nc] & 1          # bit 0 = today, even if bit 1 is set
            if live == 3 or (live == 2 and board[r][c] & 1):
                board[r][c] |= 2                           # tomorrow goes into bit 1
    for r in range(R):
        for c in range(C):
            board[r][c] >>= 1                              # tomorrow becomes today


blinker = [[0, 0, 0], [1, 1, 1], [0, 0, 0]]
game_of_life(blinker)
print(blinker)    # [[0, 1, 0], [0, 1, 0], [0, 1, 0]]

**Try it**
- Print the board just before the final `>>= 1` pass: `[[0, 2, 0], [1, 3, 1], [0, 2, 0]]`. You can read births (2), deaths (1) and survivors (3) straight off it.
- Replace `board[nr][nc] & 1` with `board[nr][nc]` and rerun the blinker: you get `[[0, 1, 0], [1, 0, 1], [0, 0, 0]]`. Cells already marked 2 or 3 count as two or three neighbours, so the centre "sees" 6 and dies.
- Run `game_of_life` twice on the blinker: it flips back to the horizontal line (it has period 2).
- A 2 x 2 block `[[1, 1], [1, 1]]` never changes: every cell has exactly 3 live neighbours.

#### Traversal patterns: name a line by what stays constant

**Idea:** along a "\" diagonal `r - c` stays the same; along a "/" anti-diagonal `r + c` stays the same; the ring (layer) of a cell is its distance to the nearest edge. So "walk the cells line by line" becomes "group the cells by a key" (the keys are in the table above). A transpose swaps the roles of `r` and `c`; for a non-square grid it must build a new C x R grid.

In [ ]:
def group_by(grid, key):                       # key(r, c) -> which line the cell is on
    lines = defaultdict(list)
    for r, row in enumerate(grid):
        for c, x in enumerate(row):
            lines[key(r, c)].append(x)         # row-major order inside each line
    return [lines[k] for k in sorted(lines)]


grid = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]
R, C = len(grid), len(grid[0])
print(group_by(grid, lambda r, c: r - c))      # [[4], [3, 8], [2, 7, 12], [1, 6, 11], [5, 10], [9]]
print(group_by(grid, lambda r, c: r + c))      # [[1], [2, 5], [3, 6, 9], [4, 7, 10], [8, 11], [12]]
print(group_by(grid, lambda r, c: min(r, c, R - 1 - r, C - 1 - c)))   # [[1, 2, 3, 4, 5, 8, 9, 10, 11, 12], [6, 7]]
print([list(col) for col in zip(*grid)])       # [[1, 5, 9], [2, 6, 10], [3, 7, 11], [4, 8, 12]]

**Try it**
- Reverse every *even-numbered* anti-diagonal and flatten: `[x for k, d in enumerate(group_by(grid, lambda r, c: r + c)) for x in (d[::-1] if k % 2 == 0 else d)]` gives `[1, 2, 5, 9, 6, 3, 4, 7, 10, 11, 8, 12]`, the zigzag order of Diagonal Traverse (498).
- Toeplitz check (766), "every '\' diagonal holds one value": `all(len(set(d)) == 1 for d in group_by(t, lambda r, c: r - c))` is `True` for `t = [[1, 2, 3], [4, 1, 2], [5, 4, 1]]`. Change one cell and watch it turn `False`.
- Walk anti-diagonal `d = 3` straight from the table, without grouping: `[grid[r][d - r] for r in range(max(0, d - C + 1), min(R, d + 1))]` is `[4, 7, 10]`, the same as `group_by(grid, lambda r, c: r + c)[3]`.

### Where it goes wrong

1. **Aliased rows.** `[[0] * C] * R` is R references to one row, so writing one cell writes the whole column. Fix: `[[0] * C for _ in range(R)]`, and copy with `[row[:] for row in grid]` (not `grid[:]`).
2. **Negative indexes do not crash.** `grid[-1][c]` quietly reads the last row, so a missing `0 <= nr` check gives wrong answers instead of an error: the neighbours of `(0, 0)` would include `grid[-1][0] = 9`. Check bounds *before* indexing.
3. **Rows and columns mixed up.** `grid[r][c]`, row first; `R = len(grid)`, `C = len(grid[0])`. Swap `R` and `C` in `neighbors` and, on the 3 x 4 grid, `(2, 0)` gets the neighbour `(3, 0)` (reading it raises `IndexError`) while `(1, 2)` loses `(1, 3)`. A square test grid hides this: always test one non-square grid.
4. **Transposing twice.** `for c in range(n)` swaps every pair twice, so after the row reversal `[[1, 2], [3, 4]]` becomes `[[2, 1], [4, 3]]`, a mirror. Use `range(r + 1, n)`, and after the transpose reverse each *row* (clockwise), not the row order (counter-clockwise).
5. **Spiral without the two re-checks.** A last single row or column is walked twice: `[[1, 2, 3]]` gives `[1, 2, 3, 2, 1]`.
6. **Destroying markers before using them.** In Set Matrix Zeroes, record whether row 0 and column 0 have zeros *first*, and zero them *last*. Without `col0`, `[[1, 1], [0, 1]]` gives `[[1, 1], [0, 0]]`.
7. **Overwriting a value others still need.** In Game of Life, write tomorrow into a spare bit and read `cell & 1`. Without the `& 1`, the blinker gives `[[0, 1, 0], [1, 0, 1], [0, 0, 0]]`.
8. **Staircase from the wrong corner.** From the top-left of `[[1, 4], [2, 5]]` with target 2: 1 is too small, but both moves (to 4 and to 2) go to bigger values, so there is no direction. Start top-right (or bottom-left).
9. **Negative slice stops.** `grid[bottom][right:left - 1:-1]` breaks when `left == 0`: the stop becomes −1, which means "the last item", so `[1, 2, 3, 4][3:-1:-1] == []`. Use `range(right, left - 1, -1)` or `row[left:right + 1][::-1]`.
10. **Rebinding instead of mutating.** Inside `rotate(matrix)`, `matrix = [list(t) for t in zip(*matrix[::-1])]` only rebinds the local name: the caller's `[[1, 2], [3, 4]]` is unchanged. `matrix[:] = ...` mutates it (correct, but it uses O(n²) extra memory).
11. **`zip` gives tuples.** `list(zip(*[[1, 2], [3, 4]]))[0][0] = 9` raises `TypeError`: convert each one with `list(t)`.
12. **`len(grid[0])` on an empty grid** raises `IndexError`: handle `[]` and `[[]]` before anything else.

### Edge cases to say out loud

Empty grid (`[]`, `[[]]`) · 1 x 1 · a single row · a single column · non-square R ≠ C (in-place rotation needs a square) · zeros inside the marker row or column · a target smaller or bigger than everything · a board with nothing alive.

In [ ]:
def after(fn, m):                              # run an in-place function on a copy, return the copy
    m = [row[:] for row in m]
    fn(m)
    return m


assert spiral_order([]) == [] and spiral_order([[]]) == []
assert spiral_order([[7]]) == [7]
assert spiral_order([[1, 2, 3]]) == [1, 2, 3]                  # one row: the guards stop the way back
assert spiral_order([[1], [2], [3]]) == [1, 2, 3]              # one column
assert spiral_order([[1, 2], [3, 4], [5, 6]]) == [1, 2, 4, 6, 5, 3]
assert after(rotate_clockwise, [[5]]) == [[5]]
assert after(rotate_clockwise, [[1, 2], [3, 4]]) == [[3, 1], [4, 2]]
assert after(set_zeroes, [[0, 1], [1, 1]]) == [[0, 0], [0, 1]]   # the zero is in the marker corner
assert after(set_zeroes, [[1], [0], [3]]) == [[0], [0], [0]]     # a single column
assert not search_sorted_matrix([[]], 1)
assert not search_sorted_matrix(sm, 0) and search_sorted_matrix(sm, 30) and search_sorted_matrix(sm, 18)
assert after(game_of_life, [[0, 0], [0, 0]]) == [[0, 0], [0, 0]]
print("edge cases pass")

**Try it**
- Add `assert after(rotate_by_rings, [[1, 2], [3, 4]]) == [[3, 1], [4, 2]]` and run: the ring version agrees on the smallest ring.
- Predict `after(game_of_life, [[1]])` before running (`[[0]]`: a lonely cell dies), then write it as an assert.
- Feed `after(rotate_clockwise, [[1, 2, 3], [4, 5, 6]])`: no error, but `[[3, 4, 1], [6, 5, 2]]` is not a rotation (a rotated 2 x 3 grid is 3 x 2). In-place rotation only makes sense for a square.

Asserts check the cases you thought of. Simple versions that are hard to get wrong check the ones you didn't: a walker that turns right whenever it is blocked (it needs a `visited` set, which is fine whenever O(R·C) extra space is allowed, and it has no guards to forget), two sets for the zeroes, and the one-line rotation from the map table. Run them side by side on random grids of every shape, 1 x N and N x 1 included. The walker is also the brute force to say first in the interview.

In [ ]:
def spiral_brute(grid):                        # the walker: turn right when blocked
    if not grid or not grid[0]:
        return []
    R, C = len(grid), len(grid[0])
    seen, out, r, c, d = set(), [], 0, 0, 0
    TURNS = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # right, down, left, up: clockwise order
    for _ in range(R * C):                      # one step per cell
        out.append(grid[r][c])
        seen.add((r, c))
        nr, nc = r + TURNS[d][0], c + TURNS[d][1]
        if not (0 <= nr < R and 0 <= nc < C) or (nr, nc) in seen:
            d = (d + 1) % 4                     # blocked: turn clockwise
            nr, nc = r + TURNS[d][0], c + TURNS[d][1]
        r, c = nr, nc
    return out


def zeroes_brute(m):                           # two sets instead of markers
    rows = {r for r, row in enumerate(m) for x in row if x == 0}
    cols = {c for row in m for c, x in enumerate(row) if x == 0}
    return [[0 if r in rows or c in cols else x for c, x in enumerate(row)] for r, row in enumerate(m)]


random.seed(0)
for _ in range(300):
    R, C = random.randint(1, 5), random.randint(1, 5)        # 1 x N and N x 1 included
    m = [[random.randint(0, 3) for _ in range(C)] for _ in range(R)]
    assert spiral_order(m) == spiral_brute(m), m
    assert after(set_zeroes, m) == zeroes_brute(m), m
    sq = [row[:min(R, C)] for row in m[:min(R, C)]]          # the largest square in the corner
    assert after(rotate_clockwise, sq) == [list(t) for t in zip(*sq[::-1])], sq
print("walls, markers and in-place rotation agree with the simple versions on 300 random grids")

**Try it**
- Delete the `if top <= bottom:` guard in `spiral_order` (rerun its cell), then rerun this one: the assert stops on the one-row grid `[[3, 0, 2]]` and prints it.
- Do the same with the `if left <= right:` guard: now the one-column grid `[[1], [1], [1]]` is the first to fail. Its three cells come out four times.
- Replace `zip(*sq[::-1])` with `zip(*sq)` (the transpose, not the turn): the rotation check fails on the very first grid.

### Variations

| Variation | What changes from the template | Problems |
|---|---|---|
| **Rotate counter-clockwise** | transpose, then reverse the row *order* (`m.reverse()`) | 48 (other way) |
| **Rotate 180°** | reverse the row order and reverse every row | 1886 (try all four turns) |
| **Spiral fill** | the same four walls, writing `1, 2, 3, ...` instead of reading | 59 |
| **Spiral outward from a cell** | step lengths 1, 1, 2, 2, 3, 3, ...; record only the steps that land inside | 885 |
| **Zigzag diagonals** | group by `r + c`, reverse every other group | 498 |
| **Toeplitz check** | every cell equals its up-left neighbour (`r - c` is constant) | 766 |
| **Transpose a non-square** | a new C x R grid: `[list(col) for col in zip(*grid)]` | 867 |
| **Shift the grid k steps** | flat index: the cell at `i` moves to `(i + k) % (R * C)` | 1260 |
| **Fully sorted (row-major) matrix** | one sorted array of R·C items: binary search on the flat index (see [Binary Search](05_Binary_Search_Sorting.ipynb#s09)) | 74 |
| **k-th smallest in a sorted matrix** | binary search on the value; a staircase counts the cells ≤ mid | 378 |
| **Sparse matrix multiplication** | multiply only the non-zero entries | 311 |
| **Tic-tac-toe on n x n** | +1/−1 counters per row, column, diagonal (`r == c`) and anti-diagonal (`r + c == n - 1`); a counter reaching ±n wins; O(1) per move | 348 |
| **Queens and diagonals** | `r - c` and `r + c` as set keys | 51, 52 |
| **Candy Crush** | mark the crushed cells by negating them, then compact each column bottom-up with a write pointer | 723 |
| **Rotating the Box** | slide the stones to the right end of each row (stopping at obstacles), then rotate clockwise | 1861 |
| **Game of Life on an unbounded board** | a set of live cells; count neighbours with a `Counter` | 289 follow-up |

**k-th smallest in a sorted matrix** (378). The cells ≤ v can be *counted* with a staircase in O(R + C): from the bottom-left, a cell ≤ v counts together with everything above it. That count only grows with v, so binary-search the smallest v whose count reaches k (binary search on the answer, as in [Binary Search](05_Binary_Search_Sorting.ipynb#s09)). That v is always a matrix value: if it weren't, `count(v - 1)` would be the same, and v would not be the smallest.

In [ ]:
def count_at_most(m, v):                       # staircase from the bottom-left: cells <= v
    r, c, count = len(m) - 1, 0, 0
    while r >= 0 and c < len(m[0]):
        if m[r][c] <= v:
            count += r + 1                     # this cell and everything above it
            c += 1
        else:
            r -= 1                             # too big, and so is the rest of this row
    return count


def kth_smallest(m, k):
    lo, hi = m[0][0], m[-1][-1]                # the answer is a value in [lo, hi]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_at_most(m, mid) >= k:
            hi = mid                           # mid is enough: the answer is <= mid
        else:
            lo = mid + 1                       # too few cells <= mid: the answer is bigger
    return lo


sorted_m = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]
print(kth_smallest(sorted_m, 8), [count_at_most(sorted_m, v) for v in (11, 12, 13)])   # 13 [5, 6, 8]

**Try it**
- `count_at_most(sorted_m, 14)` is also 8, but 14 is not in the matrix. It can never be the answer: 13 already reaches the same count, and the search looks for the *smallest* such v.
- Change `hi = mid` to `hi = mid - 1`: `kth_smallest(sorted_m, 6)` returns 11 instead of 12. `mid = 12` was itself the answer, and it got thrown away.
- `kth_smallest(sorted_m, 7)` and `kth_smallest(sorted_m, 8)` are both 13: duplicates take one rank each.

**Two Google favourites.** Sparse Matrix Multiplication (311): a zero adds nothing to `out[i][j] = sum(A[i][k] * B[k][j])`, so keep only the non-zero entries of each row of B and skip the zeros of A. Game of Life on an unbounded board: the board no longer fits in a grid, so keep the *set* of live cells and let each one vote for its 8 neighbours; a `Counter` of the votes is the neighbour count.

In [ ]:
def multiply_sparse(A, B):                     # A is R x K, B is K x C
    nz_B = [[(j, v) for j, v in enumerate(row) if v] for row in B]   # non-zeros of each row of B
    out = [[0] * len(B[0]) for _ in A]
    for i, row in enumerate(A):
        for k, a in enumerate(row):
            if a:                              # a zero in A meets nothing
                for j, b in nz_B[k]:
                    out[i][j] += a * b
    return out


def life_step(live):                           # live = a set of (r, c): no borders at all
    votes = Counter((r + dr, c + dc) for r, c in live
                    for dr in (-1, 0, 1) for dc in (-1, 0, 1) if dr or dc)
    return {cell for cell, k in votes.items() if k == 3 or (k == 2 and cell in live)}


print(multiply_sparse([[1, 0, 0], [-1, 0, 3]], [[7, 0, 0], [0, 0, 0], [0, 0, 1]]))   # [[7, 0, 0], [-7, 0, 3]]
print(sorted(life_step({(1, 0), (1, 1), (1, 2)})))   # [(0, 1), (1, 1), (2, 1)]

**Try it**
- Count the multiplications with a counter in the innermost loop: 3 for this example, against 2 · 3 · 3 = 18 for the plain triple loop.
- Run `life_step` twice on the blinker: you are back at `{(1, 0), (1, 1), (1, 2)}`.
- A glider walks: run `life_step` four times on `{(0, 1), (1, 2), (2, 0), (2, 1), (2, 2)}` and every cell has moved one step down and one step right.

### Say it in the interview

> "First the brute force: a walker with a visited set that turns right whenever it's blocked. That's O(R·C) time and O(R·C) extra space. The visited set is the waste: the unvisited cells always form a rectangle, so four walls describe them. I walk the top row, the right column, the bottom row and the left column, moving each wall in right after I walk it, and I re-check before the last two sides because the rectangle can shrink to a single row or column. O(R·C) time, O(1) extra space."

Then point at the two re-checks, and test a 1 x 3, a 3 x 1 and one non-square grid out loud. For the other tools, the line to point at is the one that makes in-place safe: `range(r + 1, n)` in the transpose, "flags first, borders last" in Set Matrix Zeroes, `& 1` in Game of Life.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Game of Life | `practice/simple/basics/matrices/05_game_of_life_in_place.py` | today in bit 0, tomorrow in bit 1; neighbours read `cell & 1`; shift everything at the end |
| Rotate Image | `math_geometry/rotate_image.py` · `practice/simple/basics/matrices/01_rotate_image.py` | clockwise = transpose (swap only above the diagonal) + reverse each row |
| Search a 2D Matrix II | `practice/simple/basics/matrices/04_search_2d_matrix_staircase.py` | start top-right: too big drops a column, too small drops a row; O(R + C) |
| Set Matrix Zeroes | `math_geometry/set_matrix_zeroes.py` · `practice/simple/basics/matrices/03_set_matrix_zeroes.py` | row 0 and column 0 hold the flags; remember their own zeros first, zero them last |
| Spiral Matrix | `math_geometry/spiral_matrix.py` · `practice/simple/basics/matrices/02_spiral_matrix.py` | four inclusive walls; walk a side, move its wall in; re-check before the bottom and left sides |

### Self-check

1. Why does the spiral need `if top <= bottom` before the bottom row, but no check before the right column?
<details><summary>Answer</summary>After <code>top += 1</code> there may be no rows left. The right-column loop is then simply empty (its <code>range(top, bottom + 1)</code> has nothing in it), but the bottom-row loop would walk the row we just walked, backwards. The <code>if</code> stops that. The same reasoning puts <code>if left &lt;= right</code> before the left column.</details>

2. In Set Matrix Zeroes, why is it safe to overwrite `m[r][0]` with a flag?
<details><summary>Answer</summary>We only write it when row r contains a zero, and then the whole row, including <code>m[r][0]</code>, ends up zero anyway. The one fact that writing destroys is whether column 0 itself held a zero, and that was saved in <code>col0</code> before marking began.</details>

3. Which corners can the staircase search start from, and why not the top-left?
<details><summary>Answer</summary>Top-right or bottom-left: each is the largest value of one line and the smallest of the other, so every comparison rules out a whole row or column. From the top-left both moves go to bigger values, so when the value is too small you don't know which way to go.</details>

4. Where does `(r, c)` go in a clockwise turn of an R x C grid, and how do you check it?
<details><summary>Answer</summary>To <code>(c, R-1-r)</code>, in a new C x R grid. Check two cells: <code>(0, 0)</code> must land in the top-right corner, <code>(0, R-1)</code>, and <code>(0, 1)</code> must land one row below it, <code>(1, R-1)</code>. Then try the map on a non-square grid.</details>

<a id="s22"></a>

## Math, Bits & Geometry

> An integer is a row of bits, and a row of bits is a small set: bit `v` is 1 exactly when `v` is in the set. Bit problems are set operations done a whole word at a time. Math problems replace *listing* things with *counting* them: digits by column, permutations by blocks, outcomes by states.

**Reach for it when** the problem mentions *bits*, *XOR*, *powers of two*, *"every element appears twice except one"*, *subsets of a small set* (n ≤ 20, 26 letters, 10 digits), *parity* ("an odd number of times"), *modulo 10⁹ + 7*, *gcd, primes or digits*, *points, lines and slopes*, or when n goes up to 10⁹ so nothing can be listed one at a time.

**In this repo:** `bit_manipulation/` (9 problems) · `math_geometry/` (7 here; Rotate Image, Spiral Matrix and Set Matrix Zeroes are in Matrices) · basics: `practice/simple/basics/bits/01_get_set_clear_toggle_bits.py`, `02_count_bits_and_lowest_set_bit.py`, `03_xor_tricks_single_number_missing_number.py`, `04_bitmask_subset_enumeration.py`, `05_reverse_bits_and_shifts.py` and `practice/simple/basics/math/01_gcd_lcm_euclid.py`, `02_primes_sieve_and_primality.py`, `03_fast_power_and_modular_arithmetic.py`, `04_digits_reverse_integer_and_palindrome_number.py`, `05_base_conversion_and_excel_columns.py`, `06_random_pick_and_reservoir_sampling.py` · a bit array as a class: `design/design_bitset.py`

### The picture

```text
   x = 180 =   1 0 1 1 0 1 0 0          bit i is worth 2^i; bit 0 is on the right
       bit:    7 6 5 4 3 2 1 0
   as a set:   {2, 4, 5, 7}             bit v is 1  <=>  v is in the set

          x  =  1 0 1 1 0 1 0 0
      x - 1  =  1 0 1 1 0 0 1 1         the borrow turns the lowest 1 into 0 and the 0s below it into 1s
  x & (x-1)  =  1 0 1 1 0 0 0 0         -> the lowest 1 is deleted    (repeat until 0 to count the 1s)
         -x  =  0 1 0 0 1 1 0 0         -x = ~x + 1 (8 bits shown): every bit above the lowest 1 flips
     x & -x  =  0 0 0 0 0 1 0 0         -> only the lowest 1 is kept  (= 4)

  XOR adds each column and keeps only "odd or even":  a ^ a = 0,  a ^ 0 = a,  in any order
       4 = 1 0 0
       1 = 0 0 1          4 ^ 1 ^ 2 ^ 1 ^ 2: in every column the pairs cancel,
       2 = 0 1 0          and the parity row that is left, 1 0 0, is the loner 4
       1 = 0 0 1
       2 = 0 1 0
     xor = 1 0 0
```

Why it is fast: a set of small integers (letters, digits, up to ~60 items) fits in one word, so "add", "is it in?", union, intersection, subset test and "flip the parity" are each one operation instead of a loop. The math tools win the same way, by **counting instead of listing**: the sieve crosses off multiples instead of testing every number; fast power squares instead of multiplying n times; digit problems work per column instead of per number; the k-th permutation and the k-th lexicographic number skip whole blocks whose size they can compute.

### From idea to code

This section is a toolbox. Its main template is the one that "bit array" interview questions are built on: a set of small integers stored as bits. The other tools follow as shorter recipes.

**The idea in one sentence:** *to remember which small integers you have seen, keep one bit per possible value: test bit `v` to ask, set bit `v` to add.*

| Decision | Bit-array answer |
|---|---|
| **State**: what must I remember? | one integer `seen` used as a set (or a `bytearray` when values go into the millions) |
| **Definition**: what exactly does each variable mean? | `(seen >> v) & 1` is 1 exactly when value `v` has appeared so far |
| **Invariant**: what is true at the end of every step? | after reading `nums[i]`, `seen` holds exactly the values in `nums[0..i]` |
| **Step**: how does one item change the state? | test bit `v`, *then* add it: `seen \|= 1 << v` |
| **Record**: when is the answer updated? | when the test finds the bit already set (a repeat); missing values are read from the bits still 0 at the end |
| **Init**: starting values | `seen = 0`, the empty set |
| **Return**: what comes back? | the repeat and/or the missing values; `full & ~seen` is "everything not seen" in one step |

The same idea, sentence by sentence:

| In words | In code |
|---|---|
| "is v in the set?" (get bit v) | `(seen >> v) & 1` |
| "add v" (set bit v) | `seen \|= 1 << v` |
| "remove v" (clear bit v) | `seen &= ~(1 << v)` |
| "flip v" (toggle bit v) | `seen ^= 1 << v` |
| "the set {0, 1, ..., n-1}" | `(1 << n) - 1` |
| "how many are in it" | `seen.bit_count()` (Python 3.10+) or `bin(seen).count("1")` |
| "its smallest member" | `(seen & -seen).bit_length() - 1` |
| "drop its smallest member" | `seen &= seen - 1` |
| "a is a subset of b" | `(a & ~b) == 0`: nothing of a lies outside b |
| "union, intersection, difference" | `a \| b`, `a & b`, `a & ~b` |
| "value i in a big bit array" | byte `i >> 3`, bit `1 << (i & 7)` |

In [ ]:
def find_error_nums(nums):                     # Set Mismatch: 1..n, one value twice, one missing
    n, seen, dup = len(nums), 0, -1            # STATE: bit v of seen = "v has appeared"; INIT: empty
    for v in nums:                             # STEP: one value at a time
        if (seen >> v) & 1:                    # test FIRST: is v already in the set?
            dup = v                            # RECORD: this is the second copy
        seen |= 1 << v                         # then add v
    full = (1 << (n + 1)) - 2                  # the set {1, ..., n}: bits 1..n are 1
    missing = (full & ~seen).bit_length() - 1  # the one member of full that was never seen
    return [dup, missing]                      # RETURN


print(find_error_nums([1, 2, 2, 4]))           # [2, 3]
print(find_error_nums([3, 1, 3]))              # [3, 2]

**Try it**
- Print `bin(full)`, `bin(seen)` and `bin(full & ~seen)` for `[1, 2, 2, 4]`: `0b11110`, `0b10110`, `0b1000`. A set difference is one `&`.
- Swap the test and the add (add first): every value now looks like a repeat, and `[1, 2, 2, 4]` answers `[4, 3]`.
- Run `find_error_nums([2, 2])`: `[2, 1]`. The hole can be the smallest value too.

**When the values go into the millions**, switch containers. A Python int is immutable, so `seen |= 1 << v` builds a brand-new integer as long as the largest bit, every time: perfect for 26 letters or 64 items, slow when every step copies up to 125 KB. A `bytearray` changes one byte in place: value `i` lives in byte `i >> 3` (that is `i // 8`), at bit `i & 7` (`i % 8`). A million possible values take 125 000 bytes; a Python `set` of a million ints needs over 30 MB.

In [ ]:
class BitArray:                                # a set of integers in [0, n), one bit each
    def __init__(self, n):
        self.data = bytearray((n + 7) // 8)    # n bits, rounded up to whole bytes, all 0

    def add(self, i):
        self.data[i >> 3] |= 1 << (i & 7)      # byte i // 8, bit i % 8 inside it

    def discard(self, i):
        self.data[i >> 3] &= ~(1 << (i & 7))

    def __contains__(self, i):                 # makes `i in bits` work
        return bool(self.data[i >> 3] & (1 << (i & 7)))   # is that bit of that byte set?


def first_repeat(ids, n):                      # ids are integers in [0, n)
    seen = BitArray(n)
    for x in ids:
        if x in seen:
            return x
        seen.add(x)
    return -1


print(first_repeat([5, 999_999, 42, 999_999, 5], 10**6))   # 999999
print(len(BitArray(10**6).data))                           # 125000 (bytes for a million values)

**Try it**
- Make `b = BitArray(20)`, add 0, 9 and 19, and print `list(b.data)`: `[1, 2, 8]`. Value 9 is bit 1 of byte 1; value 19 is bit 3 of byte 2.
- Then `b.discard(9)`: `9 in b` is now `False` and `19 in b` is still `True`.
- Swap the `BitArray` in `first_repeat` for a plain int (`seen |= 1 << x`) and time both on 20 000 random ids below 10**6 with `time.perf_counter()`: the int version is far slower, because every `|=` copies the whole number.

### Watch it work

Each line adds one value; the set is drawn as 8 bits, value 0 on the right.

In [ ]:
def trace_seen(nums):
    seen = 0
    for v in nums:
        note = "   <- bit already set: repeat" if (seen >> v) & 1 else ""
        seen |= 1 << v
        print(f"add {v}:   {seen:08b}{note}")
    full = (1 << (len(nums) + 1)) - 2
    print(f"full:    {full:08b}")
    print(f"missing: {full & ~seen:08b}")


trace_seen([3, 1, 4, 1, 5, 2])

**Try it**
- Run `trace_seen([2, 2, 1])` and read the missing value off the last line: bit 3, so 3.
- Add `seen.bit_count()` to the printed line: it grows by one per new value and stands still on the repeat.
- Change `- 2` to `- 1` in `full`: the missing line becomes `01000001`. Bit 0 joins the real hole, because the values start at 1 and bit 0 is never set.

### The rest of the toolbox

#### Bit tricks to type without thinking

**Idea:** `1 << i` is a lone 1 at position i: OR it in to set, AND with its complement to clear, XOR to flip, shift down and `& 1` to read. `x & (x - 1)` deletes the lowest 1, so a loop of it runs once per 1-bit. And `i >> 1` is `i` without its last bit, which lets you count bits for every number from the previous answers.

In [ ]:
def show(x, width=8):
    return format(x, f"0{width}b")             # x in binary, padded to width digits


x = 0b10110100                                 # 180, the set {2, 4, 5, 7}
print((x >> 5) & 1, (x >> 3) & 1)              # 1 0: get bit 5, get bit 3
print(show(x | (1 << 3)), show(x & ~(1 << 2)), show(x ^ (1 << 7)))   # 10111100 10110000 00110100
print(show(x & (x - 1)), show(x & -x))         # 10110000 00000100: delete / keep the lowest 1


def popcount(x):                               # Number of 1 Bits: one round per 1-bit
    count = 0
    while x:
        x &= x - 1                             # delete the lowest 1
        count += 1
    return count


def count_bits_upto(n):                        # Counting Bits: popcount of every i in 0..n
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        ans[i] = ans[i >> 1] + (i & 1)         # i is its half (i >> 1) with one bit appended
    return ans


def reverse_bits(x):                           # Reverse Bits: exactly 32 rounds
    out = 0
    for _ in range(32):
        out = (out << 1) | (x & 1)             # push x's lowest bit onto the right of out
        x >>= 1
    return out


print(popcount(180), (180).bit_count())                    # 4 4
print([v for v in range(1, 20) if (v & (v - 1)) == 0])     # [1, 2, 4, 8, 16]: one 1-bit each
print(count_bits_upto(5))                                  # [0, 1, 1, 2, 1, 2]
print(reverse_bits(1), reverse_bits(43261596))             # 2147483648 964176192

**Try it**
- Print `show(x & ~(x - 1))`: `00000100`, the same as `x & -x`, because `-x == ~(x - 1)` in two's complement.
- Replace `for _ in range(32):` with `while x:` in `reverse_bits`: `reverse_bits(1)` returns 1 instead of 2147483648. The 31 leading zeros of the input never became trailing zeros of the output.
- Print `count_bits_upto(15)`: the second half (8..15) is the first half (0..7) plus one, because those numbers are the first half with bit 3 added.
- Don't call `popcount(-1)`: in Python −1 has infinitely many 1-bits, so the loop never ends. Mask to 32 bits first: `popcount(-1 & 0xFFFFFFFF)` is 32.

**XOR as parity.** XOR remembers, for every bit column, whether it has seen an odd or even number of 1s. Values that come in pairs cancel, whatever their order, so whatever is left is the value with no partner.

In [ ]:
def single_number(nums):                       # every value twice except one
    acc = 0
    for x in nums:
        acc ^= x                               # pairs cancel: a ^ a = 0
    return acc


def missing_number(nums):                      # 0..n with exactly one value missing
    acc = len(nums)                            # n has no index of its own: start with it
    for i, x in enumerate(nums):
        acc ^= i ^ x                           # a present value cancels against its equal index
    return acc


print(single_number([4, 1, 2, 1, 2]), missing_number([3, 0, 1]))   # 4 2

**Try it**
- Run `single_number([7, 3, 7, 3, 3])`: 3. XOR keeps whatever appears an *odd* number of times, so "three times" breaks the trick.
- Start `missing_number` with `acc = 0` and run `missing_number([0, 1])`: 0 instead of 2. The value n never got a partner.
- Two loners (260): `both = single_number([1, 2, 1, 3, 2, 5])` is `3 ^ 5 = 6`, and `low = both & -both` is 2, a bit where 3 and 5 differ. XOR-ing only the numbers that have that bit, `single_number([v for v in [1, 2, 1, 3, 2, 5] if v & low])`, gives 3, and `both ^ 3` is 5.

#### Number theory: gcd, primes, fast power

**Idea.** Euclid: every common divisor of `a` and `b` also divides `a % b`, so `gcd(a, b) = gcd(b, a % b)`, and the pair shrinks fast. Primes: divisors come in pairs `d · (n / d)` and one of the two is at most √n, so trial division stops when `d * d > n`; to list *all* primes up to n, cross off multiples (the sieve), starting at `p * p` because smaller multiples already have a smaller prime factor. Fast power: write the exponent in binary, `3^13 = 3^8 · 3^4 · 3^1`, square the base once per bit, and multiply it in when the bit is 1.

**Why mod after every multiply?** `(a * b) % m == ((a % m) * (b % m)) % m`, so reducing early never changes the answer, and it keeps every number below m² (under 2⁶³ for m = 10⁹ + 7, which is what fixed-width languages need). Python never overflows, but `3 ** 10**6` has 477 122 digits, and multiplying giants is slow.

In [ ]:
def gcd(a, b):
    while b:
        a, b = b, a % b                        # (252, 105) -> (105, 42) -> (42, 21) -> (21, 0)
    return abs(a)


def lcm(a, b):
    return abs(a) // gcd(a, b) * abs(b) if a and b else 0     # divide first: smaller numbers


def is_prime(n):
    if n < 2:
        return False
    d = 2
    while d * d <= n:                          # a divisor pair always has one side <= sqrt(n)
        if n % d == 0:
            return False
        d += 1
    return True


def primes_upto(n):                            # sieve of Eratosthenes
    alive = [False, False] + [True] * (n - 1)  # alive[i]: is i still possibly prime?
    for p in range(2, math.isqrt(n) + 1):
        if alive[p]:
            for multiple in range(p * p, n + 1, p):   # smaller multiples are crossed off already
                alive[multiple] = False
    return [i for i in range(n + 1) if alive[i]]


def power_mod(base, exp, mod):
    result, base = 1 % mod, base % mod
    while exp:
        if exp & 1:                            # this bit of exp is 1: multiply its power in
            result = result * base % mod       # reduce after EVERY multiply
        base = base * base % mod               # base, base^2, base^4, base^8, ...
        exp >>= 1
    return result


print(gcd(252, 105), lcm(4, 6), is_prime(91), primes_upto(30))   # 21 12 False [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
print(power_mod(3, 13, 1000), pow(3, 13, 1000))                  # 323 323

**Try it**
- Print `(a, b)` at the top of the `gcd` loop for `gcd(55, 34)`: 8 rounds. Neighbouring Fibonacci numbers are the slowest case for Euclid, and even they finish in O(log) steps.
- In `primes_upto`, start crossing off at `2 * p` instead of `p * p` and count the assignments for n = 30: 28 instead of 24, with the same primes.
- Remove the `1 % mod` (start with `result = 1`) and run `power_mod(5, 0, 1)`: 1 instead of 0. Everything is 0 modulo 1, even an empty product.
- Compare `power_mod(3, 10**6, 10**9 + 7)` with `3 ** 10**6 % (10**9 + 7)`: the same answer, but the second first builds a 1.6-million-bit number. The built-in `pow(3, 10**6, 10**9 + 7)` does the fast version for you.

#### Digits and bases

**Idea:** `n % 10` is the last digit and `n // 10` drops it, so digits come out lowest first, and `rev * 10 + d` pushes a digit onto the right end of `rev`. Every base works the same way with `divmod(n, b)`; reading digits back is Horner's rule, `n = n * b + d`. Excel columns are base 26 with digits 1..26 and *no zero*, so shift each digit down by one before the divmod. For overflow, a 32-bit language can't compute `rev * 10 + d` and look afterwards: test *before* pushing, `rev > (limit - d) // 10`.

In [ ]:
def reverse_int(x):                            # Reverse Integer: 0 if the result leaves 32 bits
    limit = 2**31 - 1 if x >= 0 else 2**31     # the largest magnitude this sign allows
    n, rev = abs(x), 0
    while n:
        n, d = divmod(n, 10)                   # peel off the last digit
        if rev > (limit - d) // 10:            # rev * 10 + d would pass the limit: stop BEFORE
            return 0
        rev = rev * 10 + d                     # push d onto the right of rev
    return rev if x >= 0 else -rev


def to_base(n, b):                             # n >= 0, 2 <= b <= 36
    symbols = string.digits + string.ascii_lowercase
    out = []
    while True:
        n, d = divmod(n, b)
        out.append(symbols[d])
        if n == 0:
            return "".join(reversed(out))      # digits came out lowest first


def column_title(n):                           # 1 -> A, 26 -> Z, 27 -> AA
    out = []
    while n:
        n, r = divmod(n - 1, 26)               # shift 1..26 down to 0..25 first
        out.append(chr(ord("A") + r))
    return "".join(reversed(out))


def column_number(title):
    n = 0
    for ch in title:
        n = n * 26 + (ord(ch) - ord("A") + 1)  # Horner: shift one digit left, add this one
    return n


print(reverse_int(-123), reverse_int(1534236469))                # -321 0
print(to_base(255, 16), to_base(10, 2), int("ff", 16))           # ff 1010 255
print(column_title(28), column_title(701), column_number("ZY"))  # AB ZY 701

**Try it**
- Remove the `- 1` in `column_title` and run `column_title(26)`: `"BA"` instead of `"Z"`. Plain base 26 writes 26 as "10"; Excel has no zero digit, so 26 is the single digit Z.
- `reverse_int(1463847412)` is 2147483641, which fits; `reverse_int(1563847412)` would be 2147483651, which does not, so it returns 0.
- Delete the overflow test and run `reverse_int(1534236469)`: 9646324351, a number no 32-bit int can hold.
- `to_base(2026, 2)` is `"11111101010"`; `int("11111101010", 2)` reads it back as 2026.

#### Random sampling

**Idea (reservoir sampling):** to pick k items uniformly from a stream you can't store, keep the first k; after that, item i (0-based) replaces a random member with probability k/(i+1). Every item ends up in the sample with probability exactly k/n. With k = 1 this is "random pick" from a stream (Linked List Random Node, Random Pick Index).

In [ ]:
def reservoir_sample(stream, k):
    sample = []
    for i, x in enumerate(stream):
        if i < k:
            sample.append(x)                   # the first k fill the reservoir
        else:
            j = random.randrange(i + 1)        # a uniform slot in 0..i
            if j < k:                          # happens with probability k / (i + 1)
                sample[j] = x                  # replace a uniformly chosen member
    return sample


random.seed(0)
hits = Counter()
for _ in range(10_000):
    hits.update(reservoir_sample(range(5), 2))
print(sorted(hits.items()))   # [(0, 3990), (1, 3927), (2, 4004), (3, 4035), (4, 4044)]: all near 4000

**Try it**
- Change `random.randrange(i + 1)` to `random.randrange(i)` and rerun the cell: values 2, 3 and 4 now show up about 5000 times and 0 and 1 about 2500. Item 2 always gets in, since `j` can only be 0 or 1.
- Use `k = 1` on `range(3)` for 9000 runs: each value about 3000 times. That is random pick.
- `reservoir_sample(range(3), 5)`: `[0, 1, 2]`. With fewer items than k, you simply get them all.

#### Geometry: exact slopes, turns and corners

**Exact slopes** (Max Points on a Line). All points on one line through an anchor share a direction `(dx, dy)`. Floats can't be trusted as keys, but the direction reduced by the gcd, with a fixed sign, is exact and hashable: `(2, 6)` and `(-1, -3)` both become `(1, 3)`, and vertical lines are `(0, 1)` with no division at all.

In [ ]:
def slope_key(dx, dy):                         # one exact key per line direction
    g = math.gcd(dx, dy)                       # > 0 for two distinct points
    dx, dy = dx // g, dy // g
    if dx < 0 or (dx == 0 and dy < 0):         # (1, 2) and (-1, -2) are the same line
        dx, dy = -dx, -dy
    return dx, dy


def max_points(points):
    best = min(len(points), 1)
    for i, (x1, y1) in enumerate(points):      # anchor every line at its first point
        dirs = Counter(slope_key(x2 - x1, y2 - y1) for x2, y2 in points[i + 1:])
        if dirs:
            best = max(best, max(dirs.values()) + 1)     # + 1 for the anchor itself
    return best


print(slope_key(2, 6), slope_key(-1, -3), slope_key(0, -5))         # (1, 3) (1, 3) (0, 1)
print(max_points([[1, 1], [3, 2], [5, 3], [4, 1], [2, 3], [1, 4]]))   # 4

**Try it**
- Use a float key instead, `(y2 - y1) / (x2 - x1) if x2 != x1 else math.inf`, and run `max_points([[0, 0], [94911151, 94911150], [94911152, 94911151]])`: 3. The answer is 2: the two slopes differ, but they round to the same float.
- Delete the sign rule and run `max_points([[0, 0], [1, 1], [-1, -1]])`: 2 instead of 3, because `(1, 1)` and `(-1, -1)` became different keys.
- `max_points([[0, 0], [0, 1], [0, -1], [1, 5]])` is 3: a vertical line, with no special case.

**Turns** (Erect the Fence). The cross product `cross(o, a, b)` is positive when the path o → a → b turns left, negative when it turns right, 0 when it goes straight. Sort the trees; walk left to right building the lower fence on a stack, and pop the last post while the new tree makes a right turn (that post sits above the lower fence; if it is on the fence at all, the upper walk will collect it). Walk back right to left for the upper fence. Popping only on `< 0` keeps trees that stand exactly on a straight stretch of fence.

**Corners** (Perfect Rectangle). Small rectangles tile a big one exactly when (1) their areas add up to the bounding box and (2) toggling every rectangle's four corners in a set leaves exactly the four outer corners: every inner corner is shared by 2 or 4 tiles, so it switches back off.

```text
   cross(o, a, b) > 0       cross(o, a, b) < 0       cross(o, a, b) == 0
           b                  o ---- a                 o ---- a ---- b
           |                         |
     o --- a                         b
       left turn               right turn              straight

   X--------+-----X         A, B, C, D tile the box: every "+" is a corner of exactly 2 tiles,
   |   A    |  B  |         so toggling all corners switches every "+" back off and leaves
   +----+---+-----+         exactly the four X corners on
   |  C |    D    |
   X----+---------X
```

In [ ]:
def cross(o, a, b):                            # > 0 left turn, < 0 right turn, 0 straight
    return (a[0] - o[0]) * (b[1] - o[1]) - (a[1] - o[1]) * (b[0] - o[0])


def fence(trees):                              # Erect the Fence: the hull, keeping trees ON edges
    pts = sorted(map(tuple, trees))
    lower, upper = [], []
    for p in pts:                              # left to right along the bottom
        while len(lower) >= 2 and cross(lower[-2], lower[-1], p) < 0:
            lower.pop()                        # right turn: the middle tree is not on the lower fence
        lower.append(p)
    for p in reversed(pts):                    # right to left along the top
        while len(upper) >= 2 and cross(upper[-2], upper[-1], p) < 0:
            upper.pop()
        upper.append(p)
    return sorted(set(lower + upper))          # the two end trees are in both chains


def is_perfect_cover(rects):                   # Perfect Rectangle
    corners, area = set(), 0
    for x1, y1, x2, y2 in rects:
        area += (x2 - x1) * (y2 - y1)
        corners ^= {(x1, y1), (x1, y2), (x2, y1), (x2, y2)}   # toggle: shared corners cancel
    x1, y1 = min(r[0] for r in rects), min(r[1] for r in rects)
    x2, y2 = max(r[2] for r in rects), max(r[3] for r in rects)
    return corners == {(x1, y1), (x1, y2), (x2, y1), (x2, y2)} and area == (x2 - x1) * (y2 - y1)


print(fence([[1, 1], [2, 2], [2, 0], [2, 4], [3, 3], [4, 2]]))   # [(1, 1), (2, 0), (2, 4), (3, 3), (4, 2)]
print(is_perfect_cover([[1, 1, 3, 3], [3, 1, 4, 2], [3, 2, 4, 4], [1, 3, 2, 4], [2, 3, 3, 4]]))   # True

**Try it**
- `cross((0, 0), (1, 0), (1, 1))` is 1, a left turn; swap the last two points and it is −1.
- Change `< 0` to `<= 0` in both loops and run `fence([[1, 2], [2, 2], [4, 2]])`: the middle tree (2, 2) disappears. Popping on "straight" removes trees that stand on the fence.
- `is_perfect_cover([[0, 0, 1, 1], [0, 0, 1, 1], [0, 0, 2, 2]])` is `False`. Delete the area test and it says `True`: the two copies of the small square cancel each other's corners.
- Drop `[3, 1, 4, 2]` from the example and print `corners` before the return: the outer corner (4, 1) is gone, and three corners of the hole, (3, 1), (3, 2) and (4, 2), are switched on instead.

#### Counting instead of listing

When n goes up to 10⁹, find a unit whose size you can compute and count whole units.

**Number of Digit One:** count column by column. In the column worth p, the digit cycles 0..9, each value held for p numbers. Split n into `high` (digits above the column), `cur` (the column's digit) and `low` (digits below): `high` full cycles each show a 1 for p numbers, and the last, partial cycle adds p (`cur > 1`), `low + 1` (`cur == 1`) or nothing (`cur == 0`).

```text
  n = 2315, tens column (p = 10): high = 23, cur = 1, low = 5
  0000..2299: 23 full cycles, each with ten numbers showing a 1 in the tens column -> 23 * 10 = 230
  2300..2315: we are inside the 1-run (cur = 1): 2310..2315                         -> low + 1 = 6
```

**K-th Smallest in Lexicographical Order:** lexicographic order is a pre-order walk of the 10-ary tree whose node v has children 10v..10v+9. Standing on a node, count its subtree (level by level, clipped to n). If the k-th number lies past that whole subtree, skip it in one jump to the next sibling; otherwise step down to the first child.

```text
  n = 13:     1          2   3   4   5   6   7   8   9         pre-order: 1, 10, 11, 12, 13, 2, ..., 9
           /  |  \  \
         10  11  12  13                                         subtree of 1 = {1} + {10..13} = 5 numbers
```

In [ ]:
def count_digit_one(n):                        # Number of Digit One
    total, p = 0, 1                            # p = the column's place value: 1, 10, 100, ...
    while p <= n:
        high, cur, low = n // (p * 10), n // p % 10, n % p
        total += high * p                      # every full cycle shows a 1 for p numbers
        if cur > 1:
            total += p                         # the last cycle already passed its whole 1-run
        elif cur == 1:
            total += low + 1                   # inside the 1-run, which has lasted low + 1 numbers
        p *= 10
    return total


def kth_lexicographic(n, k):                   # K-th Smallest in Lexicographical Order
    def subtree_size(prefix):                  # how many numbers in 1..n start with prefix
        size, lo, hi = 0, prefix, prefix + 1
        while lo <= n:
            size += min(n + 1, hi) - lo        # this level is [lo, hi), clipped to n
            lo, hi = lo * 10, hi * 10
        return size

    cur, k = 1, k - 1                          # stand on 1; k = steps still to take
    while k:
        size = subtree_size(cur)
        if size <= k:                          # the answer is past this whole subtree
            k -= size
            cur += 1                           # jump to the next sibling
        else:                                  # the answer is inside: step down
            k -= 1
            cur *= 10
    return cur


print(count_digit_one(13), count_digit_one(2315))           # 6 1768
print(kth_lexicographic(13, 2), kth_lexicographic(13, 6))   # 10 2

**Try it**
- Check against the brute force: `sum(str(i).count("1") for i in range(2316))` is also 1768.
- Change `low + 1` to `low` and run `count_digit_one(10)`: 1 instead of 2. The number 10 itself is lost.
- Print `cur, k` at the top of each loop for `kth_lexicographic(100, 10)`: it steps down once (1 → 10), skips the subtree {10, 100} in one jump, then hops 11, 12, ... and stops at 17.
- Clip with `min(n, hi)` instead of `min(n + 1, hi)` and run `kth_lexicographic(13, 13)`: 10 instead of 9. The subtree that contains n is counted one short, as if n did not exist.

**Permutation Sequence:** in lexicographic order the n! permutations come in n blocks of (n−1)!, one block per first digit. So the 0-based rank divided by (n−1)! picks the first digit from the unused ones, the remainder is the rank inside that block, and you repeat with (n−2)!: that is writing the rank in the factorial number system.

```text
  n = 4, k = 9 -> rank r = 8 = 1·3! + 1·2! + 0·1! + 0·0!
  unused [1 2 3 4] take index 1 -> 2;  [1 3 4] take index 1 -> 3;  [1 4] take index 0 -> 1;  [4] -> 4:  "2314"
```

**Poor Pigs:** with T = minutesToTest // minutesToDie rounds, a pig ends in one of T + 1 states (dies in round 1, 2, ..., T, or survives), so p pigs can tell apart (T + 1)^p buckets. Give each bucket a p-digit base-(T+1) label and let pig i drink, in round r, every bucket whose i-th digit is r.

In [ ]:
def kth_permutation(n, k):                     # Permutation Sequence
    pool, out, r = list(range(1, n + 1)), [], k - 1    # r = 0-based rank
    for i in range(n - 1, -1, -1):
        idx, r = divmod(r, math.factorial(i))  # each choice of the next digit owns i! orders
        out.append(str(pool.pop(idx)))
    return "".join(out)


def poor_pigs(buckets, minutes_to_die, minutes_to_test):
    states = minutes_to_test // minutes_to_die + 1     # dies in round 1..T, or survives
    pigs = 0
    while states ** pigs < buckets:            # p pigs tell apart states ** p buckets
        pigs += 1
    return pigs


print(kth_permutation(3, 3), kth_permutation(4, 9))    # 213 2314
print(poor_pigs(4, 15, 15), poor_pigs(1000, 15, 60))   # 2 5

**Try it**
- Print `idx` at each step of `kth_permutation(4, 9)`: 1, 1, 0, 0, the factorial-base digits of 8.
- Forget the `- 1` (`r = k`): `kth_permutation(3, 3)` returns `"231"`, the 4th permutation, and `kth_permutation(3, 6)` crashes with `IndexError`.
- `poor_pigs(125, 1, 4)` is 3 (5³ = 125 exactly) and `poor_pigs(126, 1, 4)` is 4. Counting only "dead or alive" (2 states) would need 7 pigs for 125 buckets.

### Where it goes wrong

1. **Operator precedence.** In Python `&`, `|` and `^` bind tighter than `==`, so `x & 1 == 0` works (unlike C and Java). But `<<` and `>>` bind *looser* than `+` and `-`: `1 << k - 1` is `1 << (k - 1)`, and `x & mask + 1` is `x & (mask + 1)`. Parenthesise every bit expression.
2. **`~x` is negative in Python.** Ints have no fixed width, so `~0b1010 == -11`, not `0b0101`. For a k-bit complement use `x ^ ((1 << k) - 1)`.
3. **Negative numbers never run out of bits.** `-1 >> 1 == -1`, so `while x: x >>= 1` never ends for a negative x, and neither does the `x &= x - 1` loop. Mask first: `x & 0xFFFFFFFF` is the 32-bit pattern.
4. **Stopping a fixed-width loop early.** Reverse Bits must run exactly 32 rounds; stopping when `x == 0` loses the leading zeros that should become trailing zeros.
5. **A huge int as a bitset.** `seen |= 1 << v` copies the whole integer each time; past a few thousand bits, use a `bytearray`.
6. **Floats for exact questions.** Slopes, square roots and "is it collinear" need integers: reduced `(dx, dy)` pairs, cross products, `math.isqrt`.
7. **Reducing mod only at the end.** `ans = ans * x % MOD` after *every* multiply (and every addition in a sum).
8. **Off by one in 1-indexed systems.** Excel columns (`n - 1` before the divmod), the k-th permutation (`k - 1`), and "primes less than n" versus "up to n".
9. **Testing after adding.** In the bit-array template, test bit v *before* you set it, or every value looks like a repeat.

### Edge cases to say out loud

0 and 1 (no set bits, one bucket, `is_prime(1)`) · negative numbers (two's complement in Python) · the 32-bit limits (2³¹ − 1 and −2³¹) · the missing value is 0 or n · a vertical line (dx = 0) · `mod = 1` · k = 1 and k = the last rank.

In [ ]:
assert find_error_nums([2, 2]) == [2, 1] and find_error_nums([1, 1]) == [1, 2]
assert popcount(0) == 0 and popcount(2**32 - 1) == 32
assert reverse_bits(0) == 0 and reverse_bits(2**32 - 1) == 2**32 - 1
assert single_number([-3, 7, -3]) == 7                      # negatives cancel too
assert missing_number([0]) == 1 and missing_number([1]) == 0
assert gcd(0, 5) == 5 and lcm(0, 5) == 0 and gcd(-12, 18) == 6
assert not is_prime(1) and is_prime(2) and primes_upto(1) == []
assert power_mod(5, 0, 1) == 0 and power_mod(-2, 3, 5) == 2  # Python's % is never negative
assert reverse_int(-2**31) == 0 and reverse_int(0) == 0 and reverse_int(120) == 21
assert column_title(1) == "A" and column_title(52) == "AZ" and column_number("AAA") == 703
assert max_points([[0, 0]]) == 1 and max_points([[0, 0], [0, 1], [0, -1]]) == 3
assert fence([[0, 0]]) == [(0, 0)]
assert count_digit_one(0) == 0 and kth_lexicographic(1, 1) == 1
assert kth_permutation(1, 1) == "1" and kth_permutation(9, math.factorial(9)) == "987654321"
assert poor_pigs(1, 1, 1) == 0
print("edge cases pass")

**Try it**
- Predict, then add: `assert reverse_int(-1463847412) == -2147483641` (negative numbers get one more unit of room, 2³¹).
- What should `missing_number([])` return? Write the assert before running it (0: the range 0..0 lost its only value).
- Add `assert count_digit_one(1000) == 301`: the leading 1 of 1000 counts once, the 300 others come from the three lower columns.

### Variations

The bit-array template is the root of a family: change what a bit means, or what you compare masks with.

| Variation | What changes from the template | Problems |
|---|---|---|
| **Parity set** | toggle instead of set (`^=`): bit d means "d has been seen an odd number of times" | 1542, 1371 |
| **Prefix masks + hash map** | store the earliest index of each prefix mask; a substring's mask is the XOR of two prefix masks | 1542, 1915 |
| **Letter sets** | a word becomes a 26-bit mask; subset test `(w & ~p) == 0` | 1178, 318 |
| **Enumerate submasks** | `sub = (sub - 1) & mask` visits every subset of `mask` | 1178 |
| **All subsets of n items** | `for mask in range(1 << n)`: bit i says whether item i is in | 78 |
| **Maximum XOR** | a binary trie of the numbers; greedily take the opposite bit from the top | 421, 1707 |
| **XOR as parity** | pairs cancel: the single number, the missing number, two singles | 136, 268, 260 |
| **Greedy on the low bits** | read the last two bits to choose the move | 397 |
| **Huge universe** | a `bytearray`: byte `i >> 3`, bit `i & 7` | 2166 |

**Letter sets and their subsets** (1178). A word is valid for a puzzle when the word's letter *set* is a subset of the puzzle's and contains the puzzle's first letter. Words with the same letter set are interchangeable, so count each mask once. Then flip the loop: a 7-letter puzzle has only 2⁷ = 128 letter subsets, so enumerate those and look each one up, instead of testing every word.

In [ ]:
def letter_mask(word):
    m = 0
    for ch in word:
        m |= 1 << (ord(ch) - ord("a"))         # bit 0 = 'a', ..., bit 25 = 'z'
    return m


def submasks(mask):                            # every subset of mask, largest first, 0 last
    sub = mask
    while True:
        yield sub
        if sub == 0:
            return
        sub = (sub - 1) & mask                 # the next smaller subset


def valid_word_counts(words, puzzles):
    count = Counter(letter_mask(w) for w in words)
    out = []
    for p in puzzles:
        first = 1 << (ord(p[0]) - ord("a"))
        out.append(sum(count[s] for s in submasks(letter_mask(p)) if s & first))
    return out


print([[x for i, x in enumerate("abc") if (mask >> i) & 1] for mask in range(1 << 3)])
# [[], ['a'], ['b'], ['a', 'b'], ['c'], ['a', 'c'], ['b', 'c'], ['a', 'b', 'c']]
print([bin(s) for s in submasks(0b1011)])   # ['0b1011', '0b1010', '0b1001', '0b1000', '0b11', '0b10', '0b1', '0b0']
print(valid_word_counts(["aaaa", "asas", "able", "ability", "actt", "actor", "access"],
                        ["aboveyz", "abrodyz", "abslute", "absoryz", "actresz", "gaswxyz"]))   # [1, 1, 3, 2, 4, 0]

**Try it**
- Replace `(sub - 1) & mask` with `sub - 1`: `submasks(0b1011)` now yields all 12 numbers from 11 down to 0, including ones like `0b100` that are not subsets.
- Move the `if sub == 0: return` check above the `yield`: the empty set 0 is never produced (7 values instead of 8).
- Drop the `if s & first` filter: the last puzzle, `"gaswxyz"`, answers 2 instead of 0, because `"aaaa"` and `"asas"` fit its letters but lack its first letter `g`.
- `len(list(submasks(letter_mask("abcdefg"))))` is 128: why the puzzle side is the cheap side to enumerate.

**Parity masks over prefixes** (1542). A digit string can be rearranged into a palindrome when at most one digit appears an odd number of times. Keep a 10-bit mask of the parities of the prefix read so far. The parities of a substring are the XOR of two prefix masks, so for each end `j` you want the *earliest* earlier prefix whose mask equals this one (all counts even) or differs in exactly one bit (one odd digit): 11 dictionary lookups.

In [ ]:
def longest_awesome(s):
    first = {0: -1}                            # mask -> earliest prefix end with it (-1: empty prefix)
    mask = best = 0
    for j, ch in enumerate(s):
        mask ^= 1 << int(ch)                   # flip the parity of this digit
        for target in [mask] + [mask ^ (1 << d) for d in range(10)]:   # all even, or one odd
            if target in first:
                best = max(best, j - first[target])
        first.setdefault(mask, j)              # keep only the EARLIEST index
    return best


print(longest_awesome("3242415"), longest_awesome("213123"), longest_awesome("12345678"))   # 5 6 1

**Try it**
- Print `format(mask, "04b")` after each digit of `"213123"`: `0100 0110 1110 1100 1000 0000`. Every digit appears twice, so the full string's mask is 0 and it matches the seed `-1`: length 6.
- Replace `first.setdefault(mask, j)` with `first[mask] = j` (keep the latest index): `longest_awesome("1111")` drops from 4 to 2. Each repeat of a mask drags its partner index to the right, and later substrings get measured from there.
- Start with `first = {}` instead of `{0: -1}`: `longest_awesome("213123")` drops from 6 to 5, because the whole string needs the empty prefix as its partner.

**Maximum XOR with a limit** (1707). To maximise `x ^ y`, decide bits from the top: a 1 at bit 29 is worth more than all lower bits together, so at each level go to the child holding the *opposite* of x's bit when it exists. A binary trie of the numbers answers that in 30 steps (the Tries section builds tries in general). The limit `y <= m` is handled offline: sort the queries by m and insert numbers in increasing order, so the trie always holds exactly the allowed numbers.

In [ ]:
def max_xor_with_limit(nums, queries, bits=30):
    root = {}                                  # binary trie: each node is {bit: child node}

    def insert(v):
        node = root
        for b in range(bits - 1, -1, -1):      # from the top bit down
            node = node.setdefault((v >> b) & 1, {})

    def best_xor(x):
        node, out = root, 0
        for b in range(bits - 1, -1, -1):
            want = 1 - ((x >> b) & 1)          # the opposite bit makes bit b of x ^ y a 1
            if want in node:
                out |= 1 << b
                node = node[want]
            else:
                node = node[1 - want]          # forced: only one branch exists
        return out

    nums, ans, i = sorted(nums), [-1] * len(queries), 0
    for qi in sorted(range(len(queries)), key=lambda q: queries[q][1]):   # smallest limit first
        x, limit = queries[qi]
        while i < len(nums) and nums[i] <= limit:   # now the trie holds exactly nums <= limit
            insert(nums[i])
            i += 1
        if i:
            ans[qi] = best_xor(x)              # answer in the ORIGINAL query position
    return ans


print(max_xor_with_limit([0, 1, 2, 3, 4], [[3, 1], [1, 3], [5, 6]]))   # [3, 3, 7]

**Try it**
- Answer `[5, 6]` by hand first: the trie holds 0..4, `5 = 101`, and the best partner is `2 = 010`, so `5 ^ 2 = 111 = 7`.
- Answer the queries in input order instead of sorted (`for qi in range(len(queries)):`) and run `max_xor_with_limit([0, 1, 2, 3, 4], [[5, 6], [3, 1]])`: `[7, 7]` instead of `[7, 3]`. The second query sees numbers above its limit that the first query inserted.
- Set `bits=2` and rerun the main example: `[3, 3, 3]`. The top bit of 4 and 5 is never looked at, so `5 ^ 2 = 7` is out of reach.
- `max_xor_with_limit([5], [[1, 2]])` is `[-1]`: no number is allowed, so the trie stays empty.

**Greedy on the low bits** (397: even → halve; odd → add or subtract 1; reach 1 in the fewest steps). Halving just drops the last bit, so the cost is in clearing 1-bits. For odd n, look at the last two bits: `...01` → subtract (it leaves `...00`, two free halvings), `...11` → add (the carry wipes out the whole run of 1s at once). The single exception is n = 3, where 3 → 2 → 1 beats 3 → 4 → 2 → 1.

In [ ]:
def integer_replacement(n):
    steps = 0
    while n > 1:
        if n % 2 == 0:
            n //= 2                            # even: halving is the only move
        elif n == 3 or (n & 3) == 1:           # ends in 01 (or is 3): subtract
            n -= 1
        else:                                  # ends in 11: add; the carry clears the run of 1s
            n += 1
        steps += 1
    return steps


print(integer_replacement(8), integer_replacement(7), integer_replacement(2**31 - 1))   # 3 4 32

**Try it**
- Print `bin(n)` at every step for 15: `0b1111 → 0b10000 → 0b1000 → 0b100 → 0b10 → 0b1`, 5 steps. One `+ 1` turned four 1s into a single 1.
- Remove `n == 3 or`: `integer_replacement(3)` becomes 3 instead of 2 (3 → 4 → 2 → 1).
- Compare with a brute force that tries both moves at every odd number, `f(n) = 1 + min(f(n + 1), f(n - 1))`, for n in 1..200: always equal.

### Say it in the interview

> "I need to remember which values I've seen, and they're small integers, so instead of a hash set I'll use a bit array: bit v is 1 if v has appeared. Testing is `(seen >> v) & 1` and adding is `seen |= 1 << v`, both O(1). The brute force rescans the array for each value, which is O(n²); this is one pass, O(n) time and n bits of memory. If the range were in the millions I'd keep the bits in a `bytearray`: byte `v >> 3`, bit `v & 7`."

Point at the line that tests *before* it sets, and say what a bit means out loud ("bit v of `seen` means v appeared"). For the math tools, name the constraint that rules out listing (n up to 10⁹), then the unit you count instead: a digit column, a subtree, a block of (n−1)!.

### Problem map

| Problem | Where | Key insight |
|---|---|---|
| Counting Bits | `bit_manipulation/counting_bits.py` | `ans[i] = ans[i >> 1] + (i & 1)`: i is its half with one bit appended |
| Erect the Fence | `math_geometry/erect_the_fence.py` | sort, build lower and upper chains on a stack; pop only on a right turn (cross < 0) so edge trees stay |
| Excel Sheet Column Title | `practice/simple/basics/math/05_base_conversion_and_excel_columns.py` | base 26 with digits 1..26 and no zero: `divmod(n - 1, 26)` |
| Find Longest Awesome Substring | `bit_manipulation/find_longest_awesome_substring.py` | 10-bit prefix parity mask; earliest index of the same mask or of a mask one bit away |
| Integer Replacement | `bit_manipulation/integer_replacement.py` | odd n: subtract on ...01 (and for 3), add on ...11 so the carry clears the run of 1s |
| K-th Smallest in Lexicographical Order | `math_geometry/kth_smallest_in_lexicographical_order.py` | pre-order of the 10-ary tree; skip a prefix's whole subtree if k is past it, else step down |
| Max Points on a Line | `math_geometry/max_points_on_a_line.py` | per anchor, count directions as gcd-reduced (dx, dy) with a fixed sign; no floats |
| Maximum XOR With an Element From Array | `bit_manipulation/maximum_xor_with_an_element_from_array.py` | sort queries by limit, insert nums up to it into a binary trie, take the opposite bit greedily |
| Missing Number | `bit_manipulation/missing_number.py` | XOR all indices 0..n with all values: present values cancel, the missing one stays |
| Number of 1 Bits | `bit_manipulation/number_of_1_bits.py` | `n &= n - 1` deletes the lowest 1; count the deletions |
| Number of Digit One | `math_geometry/number_of_digit_one.py` | per column p: `high * p` full cycles, plus p, `low + 1` or 0 depending on the column's digit |
| Number of Valid Words for Each Puzzle | `bit_manipulation/number_of_valid_words_for_each_puzzle.py` | Counter of word letter-masks; sum over the puzzle's 128 submasks that contain its first letter |
| Palindrome Number | `practice/simple/basics/math/04_digits_reverse_integer_and_palindrome_number.py` | reverse only the lower half of the digits and compare it with the upper half |
| Perfect Rectangle | `math_geometry/perfect_rectangle.py` | areas add up to the bounding box, and toggling all corners leaves exactly its 4 corners |
| Permutation Sequence | `math_geometry/permutation_sequence.py` | factorial number system: divmod the 0-based rank by (n−1)!, (n−2)!, ... and pop from the unused digits |
| Poor Pigs | `math_geometry/poor_pigs.py` | a pig has T + 1 outcomes; the answer is the smallest p with (T + 1)^p ≥ buckets |
| Random Pick Index | `practice/simple/basics/math/06_random_pick_and_reservoir_sampling.py` | reservoir of size 1 over the matches: keep the i-th match with probability 1/i |
| Reverse Bits | `bit_manipulation/reverse_bits.py` | peel x's lowest bit onto the right of the output exactly 32 times; leading zeros matter |
| Reverse Integer | `practice/simple/basics/math/04_digits_reverse_integer_and_palindrome_number.py` | peel digits with divmod; check `rev > (limit - d) // 10` before pushing a digit |
| Single Number | `bit_manipulation/single_number.py` | XOR everything: pairs cancel, the loner stays |
| Single Number III | `practice/simple/basics/bits/03_xor_tricks_single_number_missing_number.py` | XOR of all is a ^ b; its lowest set bit splits the numbers into two Single Number problems |

### Self-check

1. Why must the bit-array template test bit v *before* it sets it?
<details><summary>Answer</summary>Setting first makes the test always see a 1, so every value would look like a repeat. The test answers "had v appeared <em>before</em> this item?", so it has to run while the set still describes the earlier items only.</details>

2. `x & (x - 1)` and `x & -x`: which one deletes the lowest 1-bit and which one keeps only it?
<details><summary>Answer</summary><code>x - 1</code> flips the lowest 1 and every 0 below it, so <code>x &amp; (x - 1)</code> deletes exactly that 1. <code>-x</code> is <code>~x + 1</code>: it flips every bit <em>above</em> the lowest 1 and keeps the lowest 1 itself, so <code>x &amp; -x</code> keeps only that bit.</details>

3. In Max Points on a Line, why reduce `(dx, dy)` by the gcd *and* fix the sign?
<details><summary>Answer</summary>Without the gcd, <code>(2, 4)</code> and <code>(1, 2)</code> would be different keys for the same direction. Without the sign rule, <code>(1, 2)</code> and <code>(-1, -2)</code>, which point along the same line through the anchor, would also be split.</details>

4. In Maximum XOR With an Element From Array, why sort the queries by their limit?
<details><summary>Answer</summary>Each query may only use numbers ≤ its limit. In increasing limit order the allowed sets only grow, so a single trie that we keep inserting into (never deleting from) holds exactly the allowed numbers when each query is answered. The answers are written back to the queries' original positions.</details>